# The Beatles, Part C: Find, Filter, and Group

**Beatles notebooks:** [A. Clean Data](A_Beatles_Clean_Data.ipynb) · [B. Tidy Data](B_Beatles_Tidy_Data.ipynb) · **C. Find, Filter, Group** · [D. Charts and Graphs](D_Beatles_Charts_Graphs.ipynb)

**The big picture.** Now we start asking questions. Almost every claim based on data rests on two simple moves:

* **Filtering** narrows the data to the rows that meet a condition: *McCartney's songs in minor keys*, *songs labelled "Melancholy"*, *the hits on Billboard's Top 50*.
* **Grouping** compares categories by summarizing each one: *average valence per year*, *most common genres for each songwriter*, *popularity of covers vs. originals*.

Pandas makes both fast, which means you can test an idea, check it from another angle, and change your mind, all in a few lines of code.

**In this notebook you will learn to:**
* build **conditions** (Boolean masks) and use them to filter rows
* filter on numbers, on text, and on **lists** of labels
* combine conditions with `&` (and), `|` (or), and `~` (not)
* turn continuous numbers into categories with **bins** (`cut`, `qcut`)
* summarize groups with `groupby()`, `agg()`, and `crosstab()`
* count correctly in tidy data (songs vs. rows)

**Input:** `beatles_clean.pkl` and `beatles_tidy_genre.pkl` (from Notebooks A and B). **Output:** tables, which you'll turn into charts in Notebook D.

**Keep in mind:** a table of averages is not yet an answer. Always ask *how many songs* stand behind each number, which songs are outliers, and how the categories were defined. Our songwriter groups and career periods are *our* choices, not facts about the Beatles.

**Contents of this Notebook**

- **1.** Load the Clean and Tidy Data
- **2.** A Basic Filter: Conditions and Boolean Masks
- **3.** Filter on Numbers
- **4.** Filtering on missing values
- **5.** Filters on Text
- **6.** Combining Conditions
- **7.** Bins: From Continuous to Categorical
- **8.** Groupby: Split, Apply, Combine
- **9.** Grouping the Tidy (Genre) Data
- **10.** Grouping an Exploded Label Column on the Fly: Moods
- **11.** Summary

## 1. Load the Clean and Tidy Data

We load **both** Pickles, because each is right for different questions (see Notebook B):

* `beatles`: the **clean** data, one row per **song**. Use it for questions about songs.
* `beatles_tidy`: the **tidy** data, one row per **song × genre**. Use it for questions about genres.

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)

beatles = pd.read_pickle('beatles_clean.pkl')
beatles_tidy = pd.read_pickle('beatles_tidy_genre.pkl')

print('clean:', beatles.shape)
print('tidy: ', beatles_tidy.shape)

clean: (278, 49)
tidy:  (613, 50)


Our research questions (from Notebook A):

* How did the Beatles' **sound change over time**?
* Do **Lennon, McCartney, and Harrison** songs differ in genre, mood, or audio features?
* Does historical **chart success** relate to present-day **Spotify popularity**?
* Which **genres and moods** are most typical, and for whom?
* How do **covers** differ from **originals**?

## 2. A Basic Filter: Conditions and Boolean Masks

A **condition** compares every value in a column with something, and returns a Series of `True` / `False` values, one for each row. This is sometimes called a **Boolean mask**.

In [2]:
beatles['songwriter_group'] == 'Harrison'

0      False
1      False
2      False
3      False
4      False
       ...  
273    False
274    False
275    False
276    False
277    False
Name: songwriter_group, Length: 278, dtype: bool

Put the mask inside square brackets, `df[mask]`, and Pandas keeps only the rows where the mask is `True`:

In [3]:
harrison_mask = beatles['songwriter_group'] == 'Harrison'
beatles[harrison_mask][['title', 'year', 'album', 'songwriter']].head(10)

,title,year,album,songwriter
10,All Things Must Pass,1969,Anthology 3,Harrison
29,Blue Jay Way,1967,Magical Mystery Tour,Harrison
36,Circles - Esher Demo,1968,The Beatles,Harrison
51,Don't Bother Me,1963,With the Beatles,Harrison
64,For You Blue,1969,Let It Be,Harrison
84,Here Comes the Sun,1969,Abbey Road,Harrison
100,I Me Mine,1970,Let It Be,Harrison
101,I Need You,1965,Help!,Harrison
107,I Want to Tell You,1966,Revolver,Harrison
125,If I Needed Someone,1965,Rubber Soul,Harrison


Note that we can also `invert` the Boolean, keeping only those rows were our test condition is FALSE.  The key here is to put our Boolean filter inside `()` and preface it with `~`:


In [4]:
harrison_mask = beatles['songwriter_group'] == 'Harrison'
beatles[~(harrison_mask)][['title', 'year', 'album', 'songwriter']].head(10)

,title,year,album,songwriter
0,12-bar Original,1965,Anthology 2,"Lennon, McCartney, Harrison and Starkey"
1,A Day in the Life,1967,Sgt. Pepper's Lonely Hearts Club Band,Lennon and McCartney
2,A Hard Day's Night,1964,A Hard Day's Night,Lennon
3,A Shot of Rhythm and Blues,1963,Live at the BBC,Thompson
4,A Taste of Honey,1963,Please Please Me,"Scott, Marlow"
5,Across the Universe,1968,Let It Be,Lennon
6,Act Naturally,1965,Help!,"Russell, Morrison"
7,Ain't She Sweet,1961,Anthology 1,"Yellen, Ager"
8,All I've Got to Do,1963,With the Beatles,Lennon
9,All My Loving,1963,With the Beatles,McCartney


Most people write these filtered views in one line:

In [5]:
beatles[beatles['songwriter_group'] == 'Harrison'].shape

(27, 49)

#### Comparison operators for Filters

| Operator | Meaning | Example |
|---|---|---|
| `==` | equal to | `beatles['year'] == 1965` |
| `!=` | not equal to | `beatles['album'] != 'Unreleased'` |
| `>` / `>=` | greater than (or equal) | `beatles['popularity'] > 70` |
| `<` / `<=` | less than (or equal) | `beatles['energy'] < 0.3` |



### Important:  use `df.copy()` when working with filtered views!


> **Saving a filtered DataFrame:** add `.copy()` when you store a filtered result you plan to modify, e.g. `harrison = beatles[harrison_mask].copy()`. This makes a truly independent DataFrame and avoids Pandas' `SettingWithCopyWarning`.



## 3. Filter on Numbers

Which songs are most popular on Spotify today?  Here we show a selected group of columns for all songs for which the Spotify `popularity` figure is greater than or equal to `x`.  Of course this comparator will only work if the given column is an integer or float!

In [7]:
beatles[beatles['popularity'] >= 75][['title', 'year', 'songwriter_group', 'popularity']].sort_values('popularity', ascending=False)

,title,year,songwriter_group,popularity
84,Here Comes the Sun,1969,Harrison,87
38,Come Together,1969,Lennon,81
140,Let It Be,1969,McCartney,81
247,Twist and Shout,1963,Cover,80
265,Yesterday,1965,McCartney,80
28,Blackbird,1968,McCartney,79
127,In My Life,1965,Lennon & McCartney,79
211,Something,1969,Harrison,76
167,Norwegian Wood (This Bird Has Flown),1965,Lennon,75


### `between` and `largest`:  Two handy shortcuts:

* `.between(low, high)` for a range (inclusive)
* `.nlargest(n, column)` / `.nsmallest(n, column)` for the top or bottom *n*

In [6]:
# very calm songs: energy between 0 and 0.25
beatles[beatles['energy'].between(0, 0.25)][['title', 'year', 'energy', 'acousticness']]

,title,year,energy,acousticness
10,All Things Must Pass,1969,0.2310,0.972
25,Because,1969,0.2490,0.754
28,Blackbird,1968,0.1270,0.754
36,Circles - Esher Demo,1968,0.1320,0.351
72,Golden Slumbers,1969,0.1520,0.326
76,Goodbye,1969,0.1740,0.918
83,Her Majesty,1969,0.2150,0.728
85,"Here, There and Everywhere",1966,0.2470,0.921
133,Julia,1968,0.0953,0.828
134,Junk,1968,0.0796,0.905


In [7]:
# the five most "danceable" Beatles songs according to Spotify
beatles.nlargest(5, 'danceability')[['title', 'year', 'danceability', 'tempo']]

,title,year,danceability,tempo
64,For You Blue,1969,0.880,128.542
172,"Ob-la-di, Ob-la-da",1968,0.818,113.059
258,Wild Honey Pie,1968,0.792,89.900
73,Good Day Sunshine,1966,0.770,117.331
13,And I Love Her,1964,0.767,113.312


## 4. Filtering on missing values

Remember that the chart columns are empty for songs that didn't chart. `.notna()` and `.isna()` make filters too. Here are the songs ranked in Billboard's Top 50 Beatles songs, best first:

In [10]:
billboard = beatles[beatles['top_50_billboard'].notna()].copy()
billboard.sort_values('top_50_billboard')[['title', 'year', 'top_50_billboard', 'highest_position_billboard', 'popularity']].head(10)

,title,year,top_50_billboard,highest_position_billboard,popularity
87,Hey Jude,1968,1.0,1.0,51
106,I Want to Hold Your Hand,1963,2.0,1.0,41
202,She Loves You,1963,3.0,1.0,51
67,Get Back,1969,4.0,1.0,70
140,Let It Be,1969,5.0,1.0,81
38,Come Together,1969,6.0,1.0,81
80,"Hello, Goodbye",1967,7.0,1.0,63
2,A Hard Day's Night,1964,8.0,1.0,71
250,We Can Work It Out,1965,9.0,1.0,41
31,Can't Buy Me Love,1964,10.0,1.0,70


We also made a True/False column for this in Notebook A. A Boolean column **is** a mask, so you can use it directly:

In [11]:
bb_hits = beatles[beatles['billboard_top50'] == True].copy()
bb_hits

,title,year,album,album_debut,songwriter,songwriter_group,lead_vocal,is_cover,genre,styles,themes,moods,popularity,duration,key,key_name,mode,mode_name,tempo,time_signature,valence,danceability,energy,loudness,acousticness,instrumentalness,liveness,speechiness,other_releases,single_a_side,single_b_side,single_certification,covered_by,billboard_top50,chart_position_uk_wikipedia,chart_position_us_wikipedia,highest_position_the_guardian,weeks_on_chart_in_uk_the_guardian,weeks_at_no1_in_uk_the_guardian,highest_position_billboard,weeks_at_no1_billboard,top_50_billboard,top_50_ultimate_classic_rock,top_50_rolling_stone,top_50_nme,top_50_top50songsorg,top_50_usa_today_2017,top_50_vulture_by_bill_wyman,uri
2,A Hard Day's Night,1964,A Hard Day's Night,UK: A Hard Day's Night US: 1962-1966,Lennon,Lennon,"Lennon, with McCartney",False,"[Rock, Electronic, Pop/Rock]","[British Invasion, Contemporary Pop/Rock, Earl...","[Everyday Life, In Love, Relationships]","[Carefree, Cheerful, Explosive, Marching, Opti...",71,152,0,C,1,major,138.514,4,0.797,0.590,0.805,-6.481,0.137000,0.000000,0.0996,0.0371,35,A Hard Day's Night; A Hard Day's Night,Things We Said Today; I Should Have Known Better,RIAA Gold,35,True,1.0,1.0,1.0,13.0,3.0,1.0,2.0,8.0,18.0,11.0,19.0,19.0,15.0,41.0,spotify:track:5J2CHimS7dWYMImCHkEFaJ
6,Act Naturally,1965,Help!,UK: Help! US: Yesterday and Today,"Russell, Morrison",Cover,Starkey,True,"[Country, Pop/Rock]","[British Invasion, Contemporary Pop/Rock, Earl...","[Hanging Out, Playful]","[Amiable/Good-Natured, Bright, Fun, Ramshackle...",52,139,7,G,1,major,93.020,4,0.944,0.702,0.447,-10.122,0.366000,0.000000,0.1530,0.0308,14,Yesterday,Act Naturally,RIAA Gold,<NA>,True,NaN,47.0,NaN,NaN,NaN,47.0,NaN,50.0,NaN,NaN,NaN,NaN,NaN,NaN,spotify:track:0IxxqsYBcCHEQ1HqLYJnwx
7,Ain't She Sweet,1961,Anthology 1,Anthology 1,"Yellen, Ager",Cover,Lennon,True,[Pop/Rock],"[British Invasion, Rock & Roll]","[Affection/Fondness, Day Driving, In Love, New...","[Rollicking, Amiable/Good-Natured, Cheerful, F...",38,133,4,E,1,major,77.967,4,0.819,0.495,0.854,-5.699,0.208000,0.000000,0.0371,0.0434,9,NaN,NaN,NaN,<NA>,True,NaN,19.0,29.0,6.0,NaN,19.0,NaN,41.0,NaN,NaN,NaN,NaN,NaN,NaN,spotify:track:3OUpVpFeNx5jeInhHqiNfs
12,All You Need Is Love,1967,Magical Mystery Tour,Magical Mystery Tour,Lennon,Lennon,Lennon,False,[Pop/Rock],"[Contemporary Pop/Rock, British Psychedelia, P...","[Affection/Fondness, Affirmation, In Love]","[Amiable/Good-Natured, Cheerful, Lush, Sentime...",68,237,7,G,1,major,103.436,4,0.653,0.400,0.480,-7.768,0.346000,0.000031,0.1550,0.0295,25,All You Need Is Love,"Baby, You're a Rich Man",RIAA Gold,20,True,1.0,1.0,NaN,NaN,NaN,1.0,NaN,15.0,23.0,21.0,24.0,7.0,33.0,NaN,spotify:track:68BTFws92cRztMS1oQ7Ewj
13,And I Love Her,1964,A Hard Day's Night,UK: A Hard Day's Night US: Something New,"McCartney, with Lennon",McCartney,McCartney,False,[Pop/Rock],"[British Invasion. Contemporary Pop/Rock, Earl...","[New Love, In Love, Relationships]","[Poignant, Reflective, Gentle, Precious, Warm,...",74,152,1,C#/Db,0,minor,113.312,4,0.636,0.767,0.331,-10.777,0.640000,0.000000,0.0681,0.0337,29,And I Love Her,If I Fell,NaN,32,True,NaN,12.0,NaN,NaN,NaN,12.0,NaN,37.0,NaN,NaN,NaN,NaN,NaN,NaN,spotify:track:65vdMBskhx3akkG9vQlSH1
21,"Baby, You're a Rich Man",1967,Magical Mystery Tour,Magical Mystery Tour,Lennon and McCartney,Lennon & McCartney,Lennon,False,"[Psychedelic Pop, Psychedelic Rock, Pop/Rock]","[British Psychedelia, Contemporary Pop/Rock, E...",[],"[Acerbic, Cynical/Sarcastic, Freewheeling, Qui...",55,183,7,G,1,major,96.751,4,0.467,0.619,0.625,-9.635,0.011900,0.000001,0.1520,0.0360,10,All You Need Is Love,"Baby, You're a Rich Man",RIAA Gold,3,True,1.0,34.0,NaN,NaN,NaN,34.0,NaN,48.0,NaN,NaN,NaN,NaN,NaN,NaN,spotify:track:0JBvtprXP2Z0LP3jmzA7Xp
31,Can't Buy Me Love,1964,A Hard Day's Night,UK: A Hard Day's Night US: Hey Jude,McCartney,McCartney,"McCartney, with Lennon",False,[Pop/Rock],"[British Invasion, Contemporary Pop/Rock, Earl...","[Awareness, In Love, Passio

## 5. Filters on Text

### Exact matches: `==` and `.isin()`

`.isin()` checks whether each value is in a **list**. It's the easiest way to select several categories at once. Here we define the Beatles' 13 core studio albums, in chronological order. We'll reuse this list later.

In [12]:
studio_albums = ['Please Please Me', 'With the Beatles', "A Hard Day's Night", 'Beatles for Sale',
                 'Help!', 'Rubber Soul', 'Revolver', "Sgt. Pepper's Lonely Hearts Club Band",
                 'Magical Mystery Tour', 'The Beatles', 'Yellow Submarine', 'Abbey Road', 'Let It Be']

beatles_studio = beatles[beatles['album'].isin(studio_albums)]
beatles_studio.shape

(186, 49)

### Negation: `~`

The tilde `~` flips a mask (True ↔ False). What's *not* on a studio album?

In [11]:
beatles[~beatles['album'].isin(studio_albums)]['album'].value_counts()

album
Live at the BBC                            30
Past Masters (Vols. 1 & 2 / Remastered)    26
Anthology 1                                19
Anthology 3                                 9
Anthology 2                                 4
On Air – Live at the BBC Volume 2           2
Rock 'n' Roll Music                         1
Unreleased                                  1
Name: count, dtype: int64

### Searching inside strings: `.str.contains()`

`.str.contains()` checks whether a **substring** appears anywhere in a text column. Which song titles mention **love**?

In [13]:
beatles[beatles['title'].str.contains('Love')]['title']

12                     All You Need Is Love
13                           And I Love Her
31                        Can't Buy Me Love
131                          It's Only Love
146                              Love Me Do
147                             Love You to
148                             Lovely Rita
179                         P.S. I Love You
187                               Real Love
202                           She Loves You
208           So How Come (No One Loves Me)
209    Soldier of Love (Lay Down Your Arms)
213        Step Inside Love / Los Paranoias
216         Sure to Fall (In Love with You)
244              To Know Her Is to Love Her
261                           Words of Love
274       You've Got to Hide Your Love Away
Name: title, dtype: object

Useful options:

* `case=False` ignores upper/lower case
* `'A|B'` means "A **or** B" (the `|` is a *regular expression*). Build it from a list with `'|'.join(my_list)`.

In [14]:
beatles[beatles['title'].str.contains('girl|woman|lady', case=False)]['title']

16                       Another Girl
69                               Girl
79                  Hello Little Girl
97                      I Got a Woman
137                      Lady Madonna
205                     She's a Woman
222                    Thank You Girl
273    You're Going to Lose That Girl
Name: title, dtype: object

> **Beware of substrings!** `str.contains('Love')` also matches `'Lovely Rita'`. That may or may not be what you want.

### Searching inside lists

Our label columns (`genre`, `styles`, `themes`, `moods`) hold **lists** (Notebook A), so the question is "is this label *in* the list?" 

Python's `in` answers exactly that, and `.apply()` asks it of every row. Unlike `.str.contains()`, this is an **exact** match: `'Rock' in labels` will not match `'Folk Rock'`.

Which songs are labelled **Melancholy**?

Here we use `apply` with a `lambda` function, which is created within this call.  In this case our lambda function means "return TRUE for any row in which the 'moods' column contains the string 'Melancholy'".  This TRUE array is then used to filter the main df of Beatles data.

In [16]:
is_melancholy = beatles['moods'].apply(lambda moods: 'Melancholy' in moods)

beatles[is_melancholy][['title', 'year', 'songwriter_group', 'moods']].head(8)

,title,year,songwriter_group,moods
5,Across the Universe,1968,Lennon,"[Atmospheric, Dreamy, Lazy, Meandering, Ramsha..."
10,All Things Must Pass,1969,Harrison,"[Autumnal, Bittersweet, Melancholy, Plaintive,..."
15,Anna (Go to Him),1963,Cover,"[Earnest, Yearning, Warm, Bittersweet, Melanch..."
20,Baby's in Black,1964,Lennon & McCartney,"[Angst-Ridden, Anguished/Distraught, Autumnal,..."
34,Cayenne,1960,McCartney,"[Earnest, Yearning, Warm, Bittersweet, Melanch..."
42,"Crying, Waiting, Hoping",1963,Cover,"[Dramatic, Innocent, Passionate, Reflective, R..."
47,Dig a Pony,1969,Lennon,"[Freewheeling, Ramshackle, Rousing, Playful, S..."
51,Don't Bother Me,1963,Harrison,"[Nocturnal, Rollicking, Angst-Ridden, Autumnal..."


Songs with *any* of several dark moods: `any(...)` is `True` if at least one of the moods in our list is in the song's list.

In [17]:
dark_moods = ['Melancholy', 'Brooding', 'Ominous', 'Gloomy', 'Menacing']

is_dark = beatles['moods'].apply(lambda moods: any(mood in moods for mood in dark_moods))
print(is_dark.sum(), 'songs')
beatles[is_dark][['title', 'year', 'songwriter_group', 'moods']].head(8)

58 songs


,title,year,songwriter_group,moods
1,A Day in the Life,1967,Lennon & McCartney,"[Ambitious, Complex, Dramatic, Dreamy, Eerie, ..."
5,Across the Universe,1968,Lennon,"[Atmospheric, Dreamy, Lazy, Meandering, Ramsha..."
10,All Things Must Pass,1969,Harrison,"[Autumnal, Bittersweet, Melancholy, Plaintive,..."
15,Anna (Go to Him),1963,Cover,"[Earnest, Yearning, Warm, Bittersweet, Melanch..."
20,Baby's in Black,1964,Lennon & McCartney,"[Angst-Ridden, Anguished/Distraught, Autumnal,..."
26,Being for the Benefit of Mr. Kite!,1967,Lennon,"[Atmospheric, Dramatic, Druggy, Eerie, Freewhe..."
29,Blue Jay Way,1967,Harrison,"[Dreamy, Eerie, Hypnotic, Quirky, Spacey, Spoo..."
34,Cayenne,1960,McCartney,"[Earnest, Yearning, Warm, Bittersweet, Melanch..."


Other string filters: `.str.startswith()`, `.str.endswith()`, `.str.len()`. For example, song titles that are questions:

In [18]:
beatles[beatles['title'].str.endswith('?')]['title']

49       Do You Want to Know a Secret?
91                   How Do You Do It?
257    Why Don't We Do It in the Road?
Name: title, dtype: object

## 6. Combining Conditions

Combine masks with:

* `&` for **and** (both must be True)
* `|` for **or** (either may be True)
* `~` for **not**

**Put each condition in parentheses!** Python evaluates `&` and `|` before `==`, so leaving out the parentheses causes confusing errors.

McCartney songs in a **minor** key:

In [19]:
beatles[(beatles['songwriter_group'] == 'McCartney') & (beatles['mode_name'] == 'minor')][['title', 'year', 'key_name', 'mode_name']]

,title,year,key_name,mode_name
9,All My Loving,1963,C#/Db,minor
13,And I Love Her,1964,C#/Db,minor
34,Cayenne,1960,D#/Eb,minor
57,Eleanor Rigby,1966,E,minor
61,Fixing a Hole,1967,C,minor
63,For No One,1966,C#/Db,minor
148,Lovely Rita,1967,G#/Ab,minor
159,Michelle,1965,F,minor
228,The End,1969,B,minor
229,The Fool on the Hill,1967,E,minor


For longer filters, it's clearer to define each mask on its own line, then combine them with the main operators:

In [20]:
is_lennon     = beatles['songwriter_group'] == 'Lennon'
is_late       = beatles['year'] >= 1967
is_reflective = beatles['moods'].apply(lambda moods: 'Reflective' in moods)

beatles[is_lennon & is_late & is_reflective][['title', 'year', 'album', 'moods']]

,title,year,album,moods
25,Because,1969,Abbey Road,"[Atmospheric, Dreamy, Reflective, Hypnotic, So..."
44,Dear Prudence,1968,The Beatles,"[Circular, Delicate, Dreamy, Ethereal, Intimat..."
119,I'm So Tired,1968,The Beatles,"[Bittersweet, Brooding, Melancholy, Nocturnal,..."
133,Julia,1968,The Beatles,"[Calm/Peaceful, Dreamy, Gentle, Intimate, Ligh..."


Which **covers** were successful enough to reach Billboard's list of top Beatles songs?

In [21]:
beatles[beatles['is_cover'] & beatles['billboard_top50']][['title', 'year', 'songwriter', 'top_50_billboard']]

,title,year,songwriter,top_50_billboard
6,Act Naturally,1965,"Russell, Morrison",50.0
7,Ain't She Sweet,1961,"Yellen, Ager",41.0
155,Matchbox,1964,"Perkins, Jefferson",40.0
207,Slow Down,1964,Williams,44.0
247,Twist and Shout,1963,"Medley, Russell",13.0


### An alternative: `.query()`

`.query()` lets you write conditions as a string. Some people find it easier to read. Use backticks for column names with spaces, and `@` to refer to a Python variable.

In [22]:
min_popularity = 70
beatles.query("songwriter_group == 'Harrison' and popularity >= @min_popularity")[['title', 'year', 'popularity']]

,title,year,popularity
84,Here Comes the Sun,1969,87
211,Something,1969,76
256,While My Guitar Gently Weeps,1968,74


## 7. Bins: From Continuous to Categorical

Sometimes a number is more useful as a **category**. "Energy = 0.83" is precise, but "high energy" is easier to group, count, and chart.

### Boolean bins

The simplest bin is a True/False column made from a condition:  here we create a new Boolean column in which all songs with `'energy' > 0.7` are TRUE.

In [23]:
beatles['high_energy'] = beatles['energy'] > 0.7
beatles['high_energy'].value_counts()

high_energy
False    199
True      79
Name: count, dtype: int64

### `pd.cut()`: bins with edges you choose

Musicologists often divide the Beatles' career into periods. With `pd.cut()` we define the **bin edges** and **labels**. Each bin includes its right edge: `(1964, 1966]` means "after 1964, up to and including 1966."

In [24]:
period_edges  = [1950, 1964, 1966, 1970, 2000]
period_labels = ['Early (to 1964)', 'Middle (1965–66)', 'Late (1967–70)', 'Reunion (1994–95)']

beatles['period'] = pd.cut(beatles['year'], bins=period_edges, labels=period_labels)
beatles['period'].value_counts(sort=False)

period
Early (to 1964)      115
Middle (1965–66)      55
Late (1967–70)       104
Reunion (1994–95)      2
Name: count, dtype: int64

### `pd.qcut()`: bins with equal numbers of songs

`pd.qcut()` chooses the edges for you, so that each bin holds (roughly) the **same number of songs**: here, thirds (*terciles*).

In [23]:
beatles['dance_level'] = pd.qcut(beatles['danceability'], q=3, labels=['low', 'medium', 'high'])
beatles.groupby('dance_level', observed=True)['danceability'].agg(['count', 'min', 'max'])

,count,min,max
dance_level,,,
low,93,0.146,0.482
medium,92,0.483,0.591
high,93,0.592,0.880


> `cut` = bins of **equal width** (or edges you choose); `qcut` = bins with **equal counts**.

## 8. Groupby: Split, Apply, Combine

`groupby()` is one of the most powerful tools in Pandas. It works in three steps:

1. **Split** the rows into groups that share a value (e.g., all songs from the same year)
2. **Apply** a calculation to each group (e.g., the mean energy)
3. **Combine** the results into a new table

The basic pattern is:

```python
df.groupby('column_to_group_by')['column(s)_to_calculate'].aggregate_function()
```

| Function | Returns, for each group |
|---|---|
| `.size()` | number of rows |
| `.count()` | number of non-missing values |
| `.nunique()` | number of distinct values |
| `.sum()`, `.mean()`, `.median()` | total, average, middle value |
| `.min()`, `.max()`, `.std()` | smallest, largest, spread |
| `.agg([...])` | several of the above at once |

### How did the Beatles' sound change over time?

In [25]:
features = ['energy', 'acousticness', 'valence', 'danceability']

beatles.groupby('year')[features].mean().round(2)

,energy,acousticness,valence,danceability
year,,,,
1958,0.41,0.99,0.54,0.54
1960,0.54,0.98,0.68,0.45
1961,0.82,0.45,0.78,0.46
1962,0.66,0.40,0.73,0.56
1963,0.63,0.50,0.86,0.56
1964,0.63,0.43,0.82,0.57
1965,0.60,0.22,0.72,0.54
1966,0.53,0.37,0.60,0.49
1967,0.51,0.27,0.58,0.48


**Questions to explore:** Which features change the most from year to year? Which stay steady? (`valence` is Spotify's measure of musical "positiveness.")

Year-by-year averages can be noisy, especially for years with only a few songs. Grouping by the **periods** we made with `pd.cut()` smooths this out:

In [26]:
beatles.groupby('period', observed=True)[features].mean().round(2)

,energy,acousticness,valence,danceability
period,,,,
Early (to 1964),0.63,0.48,0.82,0.56
Middle (1965–66),0.57,0.27,0.68,0.52
Late (1967–70),0.50,0.33,0.56,0.50
Reunion (1994–95),0.60,0.04,0.43,0.42


> `observed=True` tells Pandas to show only the categories that actually appear. It is recommended whenever you group by a *categorical* column like one made with `cut`/`qcut`.

### Do Lennon, McCartney, and Harrison songs sound different?

In [27]:
big_three = ['Lennon', 'McCartney', 'Harrison']
beatles_big_three = beatles[beatles['songwriter_group'].isin(big_three)]

beatles_big_three.groupby('songwriter_group')[features + ['popularity']].mean().round(2)

,energy,acousticness,valence,danceability,popularity
songwriter_group,,,,,
Harrison,0.53,0.3,0.56,0.46,50.59
Lennon,0.57,0.3,0.65,0.50,55.61
McCartney,0.52,0.4,0.65,0.55,57.43


### Several statistics at once: `.agg()`

`.agg()` accepts a list of functions:

In [31]:
beatles_big_three.groupby('songwriter_group')['valence'].agg(['size', 'mean']).round(2)

,size,mean
songwriter_group,,
Harrison,27,0.56
Lennon,77,0.65
McCartney,81,0.65


**Named aggregation** lets you calculate different statistics for different columns and name the results. Here is a profile of each studio album, sorted in release order:

In [28]:
album_profile = beatles_studio.groupby('album').agg(
    first_year=('year', 'min'),
    songs=('title', 'count'),
    lennon_songs=('songwriter_group', lambda s: (s == 'Lennon').sum()),
    mccartney_songs=('songwriter_group', lambda s: (s == 'McCartney').sum()),
    harrison_songs=('songwriter_group', lambda s: (s == 'Harrison').sum()),
    covers=('is_cover', 'sum'),
    mean_energy=('energy', 'mean'),
    mean_valence=('valence', 'mean'),
)
album_profile.reindex(studio_albums).round(2)

,first_year,songs,lennon_songs,mccartney_songs,harrison_songs,covers,mean_energy,mean_valence
album,,,,,,,,
Please Please Me,1962,14,4,3,0,6,0.60,0.79
With the Beatles,1963,14,3,3,1,6,0.68,0.87
A Hard Day's Night,1964,13,9,3,0,0,0.61,0.76
Beatles for Sale,1964,14,3,4,0,6,0.60,0.81
Help!,1965,14,5,5,2,2,0.61,0.73
Rubber Soul,1965,14,4,5,2,0,0.54,0.72
Revolver,1966,14,5,6,3,0,0.53,0.63
Sgt. Pepper's Lonely Hearts Club Band,1966,13,3,7,1,0,0.48,0.63
Magical Mystery Tour,1966,11,3,5,1,0,0.49,0.47


> `lambda s: (s == 'Lennon').sum()` is a tiny one-line function: for each album, it counts the songs whose `songwriter_group` is `'Lennon'`. `.reindex(studio_albums)` puts the rows in the order of our chronological list.

**Questions to explore:** What happens to the number of covers over time? How does each songwriter's share change from album to album? Do energy and valence follow the same path as the year-by-year averages above?

### Billboard Success THEN vs. Spotify popularity NOW

In [29]:
beatles.groupby('billboard_top50')['popularity'].agg(['count', 'mean', 'median']).round(1)

,count,mean,median
billboard_top50,,,
False,229,49.0,52.0
True,49,57.9,60.0


**Questions to explore:** Are the old hits still the most popular songs today? What about *within* the Top 50: are higher-ranked songs more popular? (We'll look at this with a scatter plot in Notebook D.) 

### Covers vs. originals

In [33]:
features = ['energy', 'acousticness', 'valence', 'danceability']
beatles.groupby('is_cover')[['popularity'] + features].mean().round(2)

,popularity,energy,acousticness,valence,danceability
is_cover,,,,,
False,54.49,0.55,0.35,0.64,0.52
True,37.85,0.64,0.47,0.86,0.56


**Questions to explore:** How do the covers the Beatles chose differ from their own songs? Why might covers be less (or more) popular today?

### Grouping by two columns

Pass a **list** of columns to `groupby()`. `.size()` counts the songs in each combination, and `.unstack()` turns the inner group into columns, making a readable table:

In [34]:
beatles_big_three.groupby(['period', 'songwriter_group'], observed=True).size().unstack(fill_value=0)

songwriter_group,Harrison,Lennon,McCartney
period,,,
Early (to 1964),2,23,17
Middle (1965–66),7,18,22
Late (1967–70),18,34,42
Reunion (1994–95),0,2,0


And from here we might turn this WIDE form data into LONG data for use in a chart:




In [36]:
period_writer = beatles_big_three.groupby(['period', 'songwriter_group'], observed=True).size().unstack(fill_value=0)


period_writer_long = period_writer.reset_index().melt(id_vars='period', var_name='songwriter', value_name='songs')
period_writer_long

,period,songwriter,songs
0,Early (to 1964),Harrison,2
1,Middle (1965–66),Harrison,7
2,Late (1967–70),Harrison,18
3,Reunion (1994–95),Harrison,0
4,Early (to 1964),Lennon,23
5,Middle (1965–66),Lennon,18
6,Late (1967–70),Lennon,34
7,Reunion (1994–95),Lennon,2
8,Early (to 1964),McCartney,17
9,Middle (1965–66),McCartney,22


### Working with a single group: `get_group()`

In [37]:
by_songwriter = beatles.groupby('songwriter_group')
by_songwriter.get_group('Starkey')[['title', 'year', 'album']]

,title,year,album
54,Don't Pass Me By,1968,The Beatles
173,Octopus's Garden,1969,Abbey Road


## 9. Grouping the Tidy (Genre) Data

Now let's switch to the tidy data for questions about **genres**. Since `Pop/Rock` is on nearly every song, we'll set it aside for most of these.

### Which genres are most common?

Remember to count **distinct songs** with `.nunique()`, not rows:

In [39]:
genres_no_pop_rock = beatles_tidy[beatles_tidy['genre'] != 'Pop/Rock']

genre_counts = genres_no_pop_rock.groupby('genre')['title'].nunique().sort_values(ascending=False)
genre_counts.head(15)

genre
Rock and Roll       45
Rock                39
Psychedelic Rock    28
Folk Rock           21
Hard Rock           20
R&B                 16
Blues Rock          11
Baroque Pop         10
Merseybeat          10
Folk                 9
Music Hall           8
Country              8
Psychedelic Pop      6
Power Pop            5
Rockabilly           5
Name: title, dtype: int64

### Keeping only frequent groups: `transform()`

Averages for a genre with just one or two songs aren't very meaningful. `transform()` computes a group statistic but returns it **for every row**, so we can use it as a filter:

In [40]:
genres = genres_no_pop_rock.copy()
genres['genre_songs'] = genres_no_pop_rock.groupby('genre')['title'].transform('nunique')

frequent_genres = genres[genres['genre_songs'] >= 8]
sorted(frequent_genres['genre'].unique())

['Baroque Pop',
 'Blues Rock',
 'Country',
 'Folk',
 'Folk Rock',
 'Hard Rock',
 'Merseybeat',
 'Music Hall',
 'Psychedelic Rock',
 'R&B',
 'Rock',
 'Rock and Roll']

In [35]:
frequent_genres.groupby('genre')[features + ['popularity']].mean().round(2).sort_values('acousticness')

,energy,acousticness,valence,danceability,popularity
genre,,,,,
Hard Rock,0.67,0.09,0.61,0.47,54.85
Blues Rock,0.55,0.17,0.51,0.50,59.18
Psychedelic Rock,0.57,0.17,0.53,0.45,55.14
Rock,0.60,0.25,0.71,0.54,54.13
Folk Rock,0.53,0.35,0.74,0.57,55.05
Rock and Roll,0.73,0.39,0.86,0.51,44.73
Merseybeat,0.67,0.41,0.83,0.57,46.7
R&B,0.66,0.43,0.84,0.61,45.62
Country,0.53,0.44,0.82,0.62,53.38


**Questions to explore:** Do these numbers match your ears? Which genres are most and least acoustic, energetic, or "positive"? Are the genres associated with the early years more or less popular today?

### Genre by songwriter: `pd.crosstab()`

`pd.crosstab()` is a shortcut for "group by two columns and count." Which genres did each songwriter favor?

In [44]:
big_three

['Lennon', 'McCartney', 'Harrison']

In [36]:
frequent_big_three = frequent_genres[frequent_genres['songwriter_group'].isin(big_three)]
pd.crosstab(frequent_big_three['genre'], frequent_big_three['songwriter_group'])

songwriter_group,Harrison,Lennon,McCartney
genre,,,
Baroque Pop,1,0,8
Blues Rock,3,6,2
Country,0,2,3
Folk,1,2,6
Folk Rock,2,8,8
Hard Rock,4,8,8
Merseybeat,0,2,3
Music Hall,0,0,8
Psychedelic Rock,7,12,6


### Listing the songs in each group

`.agg(list)` (or `.unique()`) collects the values in each group into a list. Which songs belong to the less common genres?

In [37]:
rare = genres[genres['genre'].isin(['Music Hall', 'Raga Rock', 'Baroque Pop', 'Country'])]
rare.groupby('genre')['title'].unique()

genre
Baroque Pop    [Eleanor Rigby, Fixing a Hole, For No One, Gol...
Country        [Act Naturally, I Forgot to Remember to Forget...
Music Hall     [Good Day Sunshine, Her Majesty, Honey Pie, Lo...
Raga Rock      [Love You to, Norwegian Wood (This Bird Has Fl...
Name: title, dtype: object

Note that we could use these lists to create a Network.  See other tutorials and notebooks!

## 10. Grouping an Exploded Label Column on the Fly: Moods

Our tidy Pickle only exploded `genre`. For a question about **moods**, we explode the `moods` lists just for this question (as recommended in Notebook B).

Which moods are most characteristic of each songwriter? `value_counts(normalize=True)` gives **proportions** rather than raw counts, which is fairer because the songwriters wrote different numbers of songs.

In [45]:
moods = beatles_big_three.explode('moods')
moods = moods.dropna(subset=['moods'])      # songs with an empty mood list explode to NaN

# now groupby on songwriters, then report normalized counts for each mood per writer

top_moods = (moods.groupby('songwriter_group')['moods']
                  .value_counts(normalize=True)
                  .groupby(level=0).head(6)       # top 6 within each songwriter
                  .round(3))
top_moods

songwriter_group  moods       
Harrison          Quirky          0.046
                  Trippy          0.043
                  Freewheeling    0.034
                  Dreamy          0.031
                  Reflective      0.031
                  Ethereal        0.028
Lennon            Rousing         0.031
                  Freewheeling    0.027
                  Quirky          0.025
                  Playful         0.023
                  Rollicking      0.022
                  Bright          0.021
McCartney         Rousing         0.040
                  Playful         0.038
                  Rollicking      0.030
                  Whimsical       0.029
                  Freewheeling    0.025
                  Cheerful        0.024
Name: proportion, dtype: float64

**Questions to explore:** Which moods do the three songwriters share? Which are distinctive for one of them? Does that fit what you know about their music? Try the same approach with `themes` or `styles`.

## 11. Summary

| Task | Code |
|---|---|
| Filter by a condition | `df[df['col'] == value]` |
| Several values | `df[df['col'].isin([a, b, c])]` |
| Text search | `df[df['col'].str.contains('A|B', case=False)]` |
| Label in a list | `df[df['col'].apply(lambda labels: 'X' in labels)]` |
| Combine | `df[(cond1) & (cond2)]`, `|` for or, `~` for not |
| Top *n* | `df.nlargest(n, 'col')` |
| Bins | `pd.cut(col, bins, labels)`, `pd.qcut(col, q, labels)` |
| Group and summarize | `df.groupby('col')['other'].mean()` |
| Several statistics | `.agg(['mean', 'count'])` or named `.agg(new=('col', 'func'))` |
| Two-way table | `groupby([a, b]).size().unstack()` or `pd.crosstab(a, b)` |
| Group statistic on every row | `groupby('col')['other'].transform('nunique')` |
| Count songs in tidy data | `.nunique()` on `title`, not `.size()` |

**Next:** in **Notebook D** we turn these tables into **charts and graphs** with Plotly Express.

**Beatles notebooks:** [A. Clean Data](A_Beatles_Clean_Data.ipynb) · [B. Tidy Data](B_Beatles_Tidy_Data.ipynb) · **C. Find, Filter, Group** · [D. Charts and Graphs](D_Beatles_Charts_Graphs.ipynb)